In [10]:
import numpy as np 
import pandas as pd 
import torch 

from sklearn.feature_extraction.text import CountVectorizer
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, random_split

import re

In [11]:
import pandas as pd
import json

# File path to your JSON file
json_file_path = 'yelp_academic_dataset_review.json'

# Initialize an empty list to store the data
data = []

# Read the JSON file line by line
with open(json_file_path, 'r') as f:
    for line in f:
        # Parse each JSON line and append it to the list
        data.append(json.loads(line))

# Convert the list of JSON objects to a DataFrame
df = pd.json_normalize(data, sep='_')

# Save the DataFrame as a CSV file
csv_file_path = 'yelp_academic_dataset_review.csv'
df.to_csv(csv_file_path, index=False)

In [5]:
import pandas as pd
import json
json_file_path = 'yelp_academic_dataset_business.json'

# Initialize an empty list to store the data
data2 = []

# Read the JSON file line by line
with open(json_file_path, 'r') as f:
    for line in f:
        # Parse each JSON line and append it to the list
        data2.append(json.loads(line))

# Convert the list of JSON objects to a DataFrame
df2 = pd.json_normalize(data2, sep='_')

# Save the DataFrame as a CSV file
csv_file_path = 'yelp_academic_dataset_business.csv'
df2.to_csv(csv_file_path, index=False)

In [7]:
df2.head(10)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attributes_AcceptsInsurance,attributes_BestNights,attributes_BYOB,attributes_Corkage,attributes_BYOBCorkage,attributes_HairSpecializesIn,attributes_Open24Hours,attributes_RestaurantsCounterService,attributes_AgesAllowed,attributes_DietaryRestrictions
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,n_0UpQx1hsNbnPUSlodU8w,Famous Footwear,"8522 Eager Road, Dierbergs Brentwood Point",Brentwood,MO,63144,38.627695,-90.340465,2.5,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,qkRM_2X51Yqxk3btlwAQIg,Temple Beth-El,400 Pasadena Ave S,St. Petersburg,FL,33707,27.766590,-82.732983,3.5,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df2.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'categories', 'hours', 'attributes_ByAppointmentOnly',
       'attributes_BusinessAcceptsCreditCards', 'hours_Monday',
       'hours_Tuesday', 'hours_Wednesday', 'hours_Thursday', 'hours_Friday',
       'hours_Saturday', 'attributes_BikeParking',
       'attributes_RestaurantsPriceRange2', 'attributes_CoatCheck',
       'attributes_RestaurantsTakeOut', 'attributes_RestaurantsDelivery',
       'attributes_Caters', 'attributes_WiFi', 'attributes_BusinessParking',
       'attributes_WheelchairAccessible', 'attributes_HappyHour',
       'attributes_OutdoorSeating', 'attributes_HasTV',
       'attributes_RestaurantsReservations', 'attributes_DogsAllowed',
       'hours_Sunday', 'attributes_Alcohol', 'attributes_GoodForKids',
       'attributes_RestaurantsAttire', 'attributes_Ambience',
       'attributes_RestaurantsTableService',
       'attribu

In [3]:
df.head(10)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15
5,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1.0,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31
6,6AxgBCNX_PNTOxmbRSwcKQ,r3zeYsv1XFBRA4dJpL78cw,gmjsEdUsKpj9Xxu6pdjH0g,5.0,0,2,0,Loved this tour! I grabbed a groupon and the p...,2015-01-03 23:21:18
7,_ZeMknuYdlQcUqng_Im3yg,yfFzsLmaWF2d4Sr0UNbBgg,LHSTtnW3YHCeUkRDGyJOyw,5.0,2,0,0,Amazingly amazing wings and homemade bleu chee...,2015-08-07 02:29:16
8,ZKvDG2sBvHVdF5oBNUOpAQ,wSTuiTk-sKNdcFyprzZAjg,B5XSoSG3SfvQGtKEGQ1tSQ,3.0,1,1,0,This easter instead of going to Lopez Lake we ...,2016-03-30 22:46:33
9,pUycOfUwM8vqX7KjRRhUEA,59MxRhNVhU9MYndMkz0wtw,gebiRewfieSdtt17PTW6Zg,3.0,0,0,0,Had a party of 6 here for hibachi. Our waitres...,2016-07-25 07:31:06


In [4]:
data = df[['text', 'stars']]

In [5]:
data['sentiment'] = ['positive' if x > 3 else 'negative' if x < 3 else 'neutral' for x in df['stars']]

/var/folders/hk/7cf1zqgd3m5dhpc25vd57jpc0000gn/T/ipykernel_86980/1704019272.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentiment'] = ['positive' if x > 3 else 'negative' if x < 3 else 'neutral' for x in df['stars']]


In [9]:
data.isnull().sum()

text         0
stars        0
sentiment    0
dtype: int64

In [8]:
data['text']= [x.lower() for x in data['text']]
data['text'] = data['text'].apply((lambda x: re.sub('[^a-zA-z0-9\s]','',x)))

/var/folders/hk/7cf1zqgd3m5dhpc25vd57jpc0000gn/T/ipykernel_86980/767771215.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['text']= [x.lower() for x in data['text']]
/var/folders/hk/7cf1zqgd3m5dhpc25vd57jpc0000gn/T/ipykernel_86980/767771215.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['text'] = data['text'].apply((lambda x: re.sub('[^a-zA-z0-9\s]','',x)))


In [10]:
data.head()

,text,stars,sentiment
0,if you decide to eat here just be aware it is ...,3.0,neutral
1,ive taken a lot of spin classes over the years...,5.0,positive
2,family diner had the buffet eclectic assortmen...,3.0,neutral
3,wow yummy different delicious our favorite...,5.0,positive
4,cute interior and owner gave us tour of upcom...,4.0,positive


In [11]:
data.shape

(6990280, 3)

In [12]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
data['sentiment_encoded'] = label_encoder.fit_transform(data['sentiment'])

/var/folders/hk/7cf1zqgd3m5dhpc25vd57jpc0000gn/T/ipykernel_86980/3775883854.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentiment_encoded'] = label_encoder.fit_transform(data['sentiment'])


In [13]:
data.head()

,text,stars,sentiment,sentiment_encoded
0,if you decide to eat here just be aware it is ...,3.0,neutral,1
1,ive taken a lot of spin classes over the years...,5.0,positive,2
2,family diner had the buffet eclectic assortmen...,3.0,neutral,1
3,wow yummy different delicious our favorite...,5.0,positive,2
4,cute interior and owner gave us tour of upcom...,4.0,positive,2


In [16]:
# Tokenization and padding
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['text'])




In [ ]:
sequences = tokenizer.texts_to_sequences(data['text'])
maxlen = 20000
X = pad_sequences(sequences, maxlen=maxlen)
y = data['sentiment_encoded']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
class YelpDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = YelpDataset(X_train, y_train)
test_dataset = YelpDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [29]:
import torch.nn as nn
import torch.optim as optim

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, bidirectional=True, dropout=0.2, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = self.dropout(lstm_out[:, -1, :])
        return self.fc(lstm_out)




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 128)          1280000   
                                                                 
 lstm (LSTM)                 (None, 100, 128)          131584    
                                                                 
 dropout (Dropout)           (None, 100, 128)          0         
                                                                 
 lstm_1 (LSTM)               (None, 128)               131584    
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense (Dense)               (None, 3)                 387       
                                                                 
Total params: 1543555 (5.89 MB)
Trainable params: 154355

KeyboardInterrupt: 

In [ ]:
vocab_size = 10000
embedding_dim = 128
hidden_dim = 128
output_dim = 3

lstm_model = LSTMModel(vocab_size, embedding_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)


In [ ]:

# Training the LSTM model
num_epochs = 5
lstm_model.train()

for epoch in range(num_epochs):
    epoch_loss = 0
    epoch_acc = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = lstm_model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += (predictions.argmax(1) == y_batch).sum().item()
    
    epoch_loss /= len(train_loader.dataset)
    epoch_acc /= len(train_loader.dataset)
    
    print(f'Epoch {epoch + 1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')

In [ ]:

# Evaluating the LSTM model
lstm_model.eval()
test_loss = 0
test_acc = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = lstm_model(X_batch)
        loss = criterion(predictions, y_batch)
        test_loss += loss.item()
        test_acc += (predictions.argmax(1) == y_batch).sum().item()

test_loss /= len(test_loader.dataset)
test_acc /= len(test_loader.dataset)
print(f'LSTM Model Test Loss: {test_loss:.4f}')
print(f'LSTM Model Test Accuracy: {test_acc:.4f}')